In [7]:
import json
import os
import  numpy as np
import pandas as pd
from keras_preprocessing.image import img_to_array

from pycocotools.coco import COCO
from PIL import Image


os.environ['SM_FRAMEWORK']='tf.keras'
import segmentation_models as sm
from tensorflow.keras import Model
from tensorflow.keras.utils import Sequence,load_img
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
import ctypes
import tensorflow as tf

In [8]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)


-2147483646

In [9]:
test_img="./arcade/stenosis/test/images/"
train_img="./arcade/stenosis/train/images/"
val_img="./arcade/stenosis/val/images/"

In [10]:
js_train="./arcade/stenosis/train/annotations/train.json"
js_val="./arcade/stenosis/val/annotations/val.json"
js_test="./arcade/stenosis/test/annotations/test.json"

with open (js_train,"r") as f:
    js_tra=json.load(f)

with open (js_val,"r") as b:
    js_v=json.load(b)

with open (js_test,"r") as c:
    js_te=json.load(c)

In [11]:
print(f" count train:{len(os.listdir(train_img))}")
print(f" count val:{len(os.listdir(val_img))}")
print(f" count test:{len(os.listdir(test_img))}")

 count train:1000
 count val:200
 count test:300


In [12]:
img_filers=os.listdir(train_img)
x_train=[]
coco=COCO(js_train)
img_ids=coco.getImgIds()
for img_id in img_ids:
    img_p=coco.loadImgs(img_id)[0]
    img=os.path.join(train_img,img_p["file_name"])
    img=load_img(img,target_size=(512,512),color_mode="grayscale")
    img=img_to_array(img)
    x_train.append(img)
x_train=np.array(x_train)

img_filers_test=os.listdir(test_img)
x_test=[]
for j in img_filers_test:
    img_p2=os.path.join(test_img,j)
    img2=load_img(img_p2,target_size=(512,512),color_mode="grayscale")
    img2=img_to_array(img2)
    x_test.append(img2)
x_test=np.array(x_test)

img_filers_val=os.listdir(val_img)
val_a=[]
for k in img_filers_val:
    img_p3=os.path.join(val_img, k)
    img3=load_img(img_p3,target_size=(512,512),color_mode="grayscale")
    img3=img_to_array(img3)
    val_a.append(img3)
val_a=np.array(val_a)



In [13]:
print("train:",x_train.shape)
print("test:",x_test.shape)
print("val:",val_a.shape)

train: (1000, 512, 512, 1)
test: (300, 512, 512, 1)
val: (200, 512, 512, 1)


In [14]:
from pycocotools.coco import COCO
import numpy as np

def make_masks(json_path):
    coco=COCO(json_path)
    img_ids=coco.getImgIds()
    masks=[]
    for img_id in img_ids:

        anns=coco.loadAnns(coco.getAnnIds(imgIds=img_id))

        mask=np.zeros((512,512))
        for ann in anns:
            o=coco.annToMask(ann)
            mask=np.maximum(mask,o)

        masks.append(mask)

    masks=np.array(masks)
    masks=np.expand_dims(masks,-1)

    return masks

In [15]:
y_train=make_masks(js_train)
y_val=make_masks(js_val)
y_test=make_masks(js_test)

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [16]:
print("val mask:",y_val.shape)
print("test mask:",y_test.shape)
print("train mask:",y_train.shape)

val mask: (200, 512, 512, 1)
test mask: (300, 512, 512, 1)
train mask: (1000, 512, 512, 1)


In [17]:
print(x_train.shape)
print(y_train.shape)

(1000, 512, 512, 1)
(1000, 512, 512, 1)


In [8]:
model=sm.Unet('resnet34',input_shape=(512,512,1),activation="sigmoid",encoder_weights=None,classes=1)

In [9]:
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])


In [ ]:
history=model.fit(x_train,y_train,validation_data=(val_a,y_val),epochs=100,batch_size=32)

In [ ]:
os.system("shutdown /s /t 60")

In [10]:
print(js_tra.keys())

print(js_tra["annotations"][0])

dict_keys(['images', 'annotations', 'categories'])
{'id': 1, 'image_id': 676, 'category_id': 26, 'segmentation': [[278.0, 291.75, 286.75, 299.25, 289.5, 296.25, 291.75, 293.25, 294.0, 290.25, 296.5, 287.5, 298.75, 284.25, 300.75, 281.25, 303.0, 278.25, 305.0, 275.0, 307.0, 271.75, 309.0, 269.0, 311.25, 265.75, 317.0, 256.0, 320.75, 248.5, 311.25, 245.12, 306.75, 251.38, 304.75, 254.62, 302.75, 257.25, 300.25, 260.25, 298.25, 263.25, 296.0, 266.25, 294.0, 269.5, 292.0, 272.0, 289.5, 275.5, 287.25, 278.5, 285.25, 281.5, 282.75, 285.0, 280.5, 287.75]], 'area': 708.0, 'bbox': [278.0, 245.12, 42.75, 54.13], 'iscrowd': 0, 'attributes': {'occluded': False}}
